# 01 — Exploratory data analysis

Inspect the actual prepared MRI dataset. First follow `DATASET.md` and run the preparation script. This notebook never downloads a dataset, invents examples, or trains a model.

This application is an academic/research prototype for brain MRI image classification. It is not a medical diagnostic device and should not be used to make medical decisions. Predictions should be reviewed by qualified medical professionals.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "config" / "config.yaml").is_file() and (path / "src").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils import load_config, resolve_path
config = load_config(PROJECT_ROOT / "config" / "config.yaml")
print(f"Project: {PROJECT_ROOT.name}; task: {config['task']}")

## Dataset availability

The manifest is the source of truth after exclusions and split auditing. Missing data is reported explicitly; run this notebook again after preparation.

In [ ]:
manifest_path = resolve_path(config["manifest_path"], config)
manifest = None
if manifest_path.is_file():
    from src.data_loader import load_manifest
    manifest = load_manifest(config)
    print(f"Eligible images: {len(manifest):,}")
    display(manifest.groupby(["split", "source_class"]).size().unstack(fill_value=0))
    display(manifest[["width", "height"]].describe())
else:
    print("Dataset preparation required — no real image data available.")

## Generate reproducible EDA artifacts

The EDA script saves class and split distributions, sample images, size/intensity summaries, and training-only augmentation examples under `results/eda/`. Example augmentation is illustrative and does not change the held-out splits.

In [ ]:
artifacts = None
if manifest is not None and not manifest.empty:
    from src.eda import run_eda
    artifacts = run_eda(config)
    print(artifacts)
else:
    print("EDA skipped: prepare a nonempty dataset first.")

In [ ]:
from IPython.display import Image, display
eda_dir = resolve_path(config.get("eda_dir", "results/eda"), config)
if artifacts is not None:
    for image_path in sorted(eda_dir.glob("*.png")):
        display(Image(filename=str(image_path), width=850))

## Interpretation checklist

Record the observed final class/split counts and exclusions in your academic report. Inspect imbalance after binary class merging; class weights must use training counts only. Check image dimensions and examples for artifacts or acquisition differences. Do not infer patient independence from numbered filenames. Do not use test images to choose augmentation strength or a prediction threshold.